# ByteMoE feasibility gates — Colab runner

Run this notebook on a **T4 GPU** (`Runtime` → `Change runtime type` → `T4 GPU`). It runs the three early go/no-go checks:

1. **E0:** all packed SwiGLU blocks reproduce the original expert and model next-token prediction.
2. **E1:** 25%/50% importance-ordered prefixes beat equally sized random prefixes.
3. **E2:** candidate blocks retain a transfer/kernel advantage over whole-expert loading.

The notebook clones [sahilaf/ByteMoE](https://github.com/sahilaf/ByteMoE) directly to `/content/ByteMoE`.

In [ ]:
# Confirm that Colab assigned a GPU. Stop here if this prints False.
import torch
assert torch.cuda.is_available(), 'Enable a T4 GPU: Runtime > Change runtime type > T4 GPU.'
print(torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
# Clone the current ByteMoE repository. Re-run this cell after a session reset.
from pathlib import Path

PROJECT_DIR = Path('/content/ByteMoE')
REPO_URL = 'https://github.com/sahilaf/ByteMoE.git'
if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

assert (PROJECT_DIR / 'requirements.txt').exists(), f'Missing requirements.txt in {PROJECT_DIR}'
%cd /content/ByteMoE
print('Project:', PROJECT_DIR)

In [ ]:
# Install the runner dependencies. The OLMoE checkpoint downloads on the first model command.
!pip -q install -r requirements.txt
!python -m pytest -q

## Model configuration

The default model is the primary OLMoE target. Change `MODEL_ID` only if you are deliberately testing another compatible MoE model. Use `fp16` on a T4.

In [ ]:
MODEL_ID = 'allenai/OLMoE-1B-7B-0924'
DTYPE = 'fp16'
EXPERT_INDEX = 0  # Update after the next cell if needed.
BLOCKS = 16

# Optional: authenticate only if Hugging Face asks for it.
# !huggingface-cli login

In [ ]:
# This prints [index] expert-path: hidden=... intermediate=... .
# Select an index that is likely to be routed by the supplied prompts (0 is a sensible first try).
!python -m scripts.e0_exactness --model {MODEL_ID} --dtype {DTYPE} --list-experts

## E0 — exactness

This must print `passed: True`. If it does not, **stop**: packing or the model adapter needs correction before approximate prefixes have meaning.

In [ ]:
!python -m scripts.e0_exactness --model {MODEL_ID} --dtype {DTYPE} --expert-index {EXPERT_INDEX} --blocks {BLOCKS} --prompt-copies 32
!cat results/e0_exactness.json

## E1 — prefix viability

This starts with a norm-based importance heuristic. It is a quick feasibility test, not the final learned predictor. Continue only when `passed: True` and the ordered prefix beats random by a material margin at 25% or 50%.

In [ ]:
!python -m scripts.e1_prefix_viability --model {MODEL_ID} --dtype {DTYPE} --expert-index {EXPERT_INDEX} --fractions 0.25 0.5 --prompt-copies 32
!cat results/e1_prefix_viability.json

## E2 — transfer/kernel microbenchmark

Copy `hidden` and `intermediate` from the expert-list output into the next cell. E2 tests pinned-memory H2D copies and SwiGLU compute at several widths plus the whole-expert reference.

In [ ]:
# Replace these with dimensions printed by the expert list.
HIDDEN_SIZE = 2048
INTERMEDIATE_SIZE = 8192
BLOCK_WIDTHS = '64 128 256 512 1024 2048'

!python -m scripts.e2_microbenchmark --hidden-size {HIDDEN_SIZE} --intermediate-size {INTERMEDIATE_SIZE} --block-widths {BLOCK_WIDTHS} --repetitions 100
!cat results/e2_microbenchmark.csv

## Decision

Continue only if all three conditions hold:

- E0: `passed: True` and top-1 agreement is 1.0.
- E1: `passed: True`; the importance prefix has at least the configured 0.05 agreement lift above random.
- E2: `aggregate/full` is below 1.0 for the block width and prefix fraction that E1 found useful.

Download the raw reports before ending the Colab session.

In [ ]:
from google.colab import files
files.download('results/e0_exactness.json')
files.download('results/e1_prefix_viability.json')
files.download('results/e2_microbenchmark.csv')